# Chapter 1.01 — PyTorch Module Building Blocks (Practice)

Read `ch01.01-pytorch-module-patterns-explanation.md` first — every exercise here practices a concept explained there. Fill in each `# YOUR CODE HERE` block yourself. Verification cells underneath each exercise are already complete; run them to check your implementation.

## Setup

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

print("torch version:", torch.__version__)

## Exercise 1 — Manual `Linear`

Implement `ManualLinear`, a from-scratch replica of `nn.Linear`.

1. In `__init__(self, in_features, out_features)`, create two `nn.Parameter` attributes:
   - `self.weight`, shape `(out_features, in_features)`, initialised with `torch.randn(...) * 0.1`
   - `self.bias`, shape `(out_features,)`, initialised with `torch.zeros(...)`
2. In `forward(self, x)`, return `x @ self.weight.T + self.bias`.

In [ ]:
class ManualLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        # YOUR CODE HERE
        pass

    def forward(self, x):
        # YOUR CODE HERE
        pass

**Verification** — copy `ManualLinear`'s weights into a real `nn.Linear` and confirm identical output.

In [ ]:
manual_layer = ManualLinear(4, 3)
torch_layer = nn.Linear(4, 3)

with torch.no_grad():
    torch_layer.weight.copy_(manual_layer.weight)
    torch_layer.bias.copy_(manual_layer.bias)

x = torch.randn(2, 4)
y_manual = manual_layer(x)
y_torch = torch_layer(x)

print("ManualLinear output:\n", y_manual)
print("nn.Linear output:\n", y_torch)
print("Match:", torch.allclose(y_manual, y_torch))

## Exercise 2 — Weight Initialization and Activation Variance

Build a 5-layer stack of `nn.Linear(20, 20, bias=False)` layers and watch how the variance of the activations grows or stays stable depending on the initialization scale.

1. Write `make_stack(num_layers, dim, init_std)` returning an `nn.ModuleList` of `num_layers` `nn.Linear(dim, dim, bias=False)` layers, each with `nn.init.normal_(layer.weight, mean=0.0, std=init_std)` applied.
2. Write `forward_through_stack(layers, x)` that passes `x` through each layer in order and returns a list of the output variance (`out.var().item()`) after each layer.

In [ ]:
def make_stack(num_layers, dim, init_std):
    # YOUR CODE HERE
    pass


def forward_through_stack(layers, x):
    # YOUR CODE HERE
    pass

**Verification** — compare `std=1.0` (too large) against `std=1/sqrt(20)` (Xavier-style).

In [ ]:
x = torch.randn(1, 20)

for label, std in [("too large (std=1.0)", 1.0), ("Xavier-style (std=1/sqrt(20))", 1.0 / (20 ** 0.5))]:
    layers = make_stack(num_layers=5, dim=20, init_std=std)
    variances = forward_through_stack(layers, x)
    print(f"\n{label}")
    for i, v in enumerate(variances):
        print(f"  after layer {i + 1}: var={v:.4f}")

## Exercise 3 — The Same MLP, Three Ways

Build one small MLP (`10 -> 20 -> 20 -> 5`, `ReLU` between layers) three different ways: `nn.Sequential`, `nn.ModuleList`, and `nn.ModuleDict`.

1. `MLPSequential(nn.Module)` — store the three `Linear`/`ReLU` layers in a single `self.layers = nn.Sequential(...)`. `forward(self, x)` just calls `self.layers(x)`.
2. `MLPModuleList(nn.Module)` — store `[Linear(10,20), ReLU(), Linear(20,20), ReLU(), Linear(20,5)]` in `self.layers = nn.ModuleList([...])`. `forward(self, x)` loops over `self.layers` and applies each one to `x`.
3. `MLPModuleDict(nn.Module)` — store the same five modules keyed by name (e.g. `"fc1"`, `"act1"`, `"fc2"`, `"act2"`, `"fc3"`) in `self.layers = nn.ModuleDict({...})`. `forward(self, x)` looks them up by name in order and applies each one.

In [ ]:
class MLPSequential(nn.Module):
    def __init__(self):
        super().__init__()
        # YOUR CODE HERE
        pass

    def forward(self, x):
        # YOUR CODE HERE
        pass


class MLPModuleList(nn.Module):
    def __init__(self):
        super().__init__()
        # YOUR CODE HERE
        pass

    def forward(self, x):
        # YOUR CODE HERE
        pass


class MLPModuleDict(nn.Module):
    def __init__(self):
        super().__init__()
        # YOUR CODE HERE
        pass

    def forward(self, x):
        # YOUR CODE HERE
        pass

**Verification** — copy weights across all three so they're identical, confirm they produce the same output, then look at the classic plain-list pitfall from the explanation (§4): a `Linear` stored on `self` inside a plain Python list is invisible to `.parameters()`.

In [ ]:
seq_model = MLPSequential()
ml_model = MLPModuleList()
md_model = MLPModuleDict()

# Copy seq_model's weights into the other two so all three are identical
linear_layers_seq = [m for m in seq_model.layers if isinstance(m, nn.Linear)]
linear_layers_ml = [m for m in ml_model.layers if isinstance(m, nn.Linear)]
linear_layers_md = [m for m in md_model.layers.values() if isinstance(m, nn.Linear)]

with torch.no_grad():
    for src, dst1, dst2 in zip(linear_layers_seq, linear_layers_ml, linear_layers_md):
        dst1.weight.copy_(src.weight)
        dst1.bias.copy_(src.bias)
        dst2.weight.copy_(src.weight)
        dst2.bias.copy_(src.bias)

x = torch.randn(2, 10)
y_seq = seq_model(x)
y_ml = ml_model(x)
y_md = md_model(x)

print("Sequential vs ModuleList match:", torch.allclose(y_seq, y_ml))
print("Sequential vs ModuleDict match:", torch.allclose(y_seq, y_md))


class BrokenList(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = [nn.Linear(4, 4) for _ in range(3)]  # plain Python list -- the bug


broken = BrokenList()
print("\nParameters visible on BrokenList:", sum(p.numel() for p in broken.parameters()))
print("(should be 0 -- the Linear layers are real but invisible to .parameters())")

## Exercise 4 — `ExampleDeepNeuralNetwork` With Optional Shortcuts

This is the pattern from the explanation (§7) and from Chapter 4 §12-13. `GELU` is provided below so you can focus on the container/shortcut logic.

1. In `ExampleDeepNeuralNetwork.__init__(self, layer_sizes, use_shortcut)`, create `self.layers`, an `nn.ModuleList` of `nn.Sequential(nn.Linear(layer_sizes[i], layer_sizes[i+1]), GELU())` blocks, one for each consecutive pair in `layer_sizes`.
2. In `forward(self, x)`, loop over `self.layers`. For each layer, compute `layer_output = layer(x)`. If `self.use_shortcut` is `True` and `x.shape == layer_output.shape`, set `x = x + layer_output` (the residual add). Otherwise set `x = layer_output`. Return `x` after the loop.
3. Write `print_gradients(model, x)`: run `output = model(x)`, compute `loss = output.mean()`, call `loss.backward()`, then for every `(name, param)` in `model.named_parameters()` where `"weight"` is in `name`, print `f"{name} has gradient mean of {param.grad.abs().mean().item()}"`.

In [ ]:
class GELU(nn.Module):
    """GPT-2's tanh-based approximation of GELU (covered in Chapter 4 SS3)."""

    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))


class ExampleDeepNeuralNetwork(nn.Module):
    def __init__(self, layer_sizes, use_shortcut):
        super().__init__()
        self.use_shortcut = use_shortcut
        # YOUR CODE HERE
        pass

    def forward(self, x):
        # YOUR CODE HERE
        pass


def print_gradients(model, x):
    # YOUR CODE HERE
    pass

**Verification** — same `layer_sizes` and `sample_input` as Chapter 4 SS12-13. Compare gradient magnitudes with and without the shortcut.

In [ ]:
layer_sizes = [3, 3, 3, 3, 3, 1]
sample_input = torch.tensor([[1., 0., -1.]])

print("Without shortcut connections:")
torch.manual_seed(123)
model_without_shortcut = ExampleDeepNeuralNetwork(layer_sizes, use_shortcut=False)
print_gradients(model_without_shortcut, sample_input)

print("\nWith shortcut connections:")
torch.manual_seed(123)
model_with_shortcut = ExampleDeepNeuralNetwork(layer_sizes, use_shortcut=True)
print_gradients(model_with_shortcut, sample_input)

## Exercise 5 — Parameter Counting Utility

Write `count_parameters(model)` (explanation §8):

1. Iterate `model.named_parameters()`. For each `(name, param)`, compute `n = param.numel()`.
2. Accumulate `total += n` always, and `trainable += n` when `param.requires_grad` is `True`.
3. Print one line per parameter: name, shape, count, and whether it's trainable.
4. After the loop, print the total and trainable counts, and return `(total, trainable)`.

In [ ]:
def count_parameters(model):
    # YOUR CODE HERE
    pass

**Verification** — run on `model_with_shortcut` from Exercise 4.

In [ ]:
total, trainable = count_parameters(model_with_shortcut)

## Exercise 6 — Bonus: `register_buffer` and `model.apply()`

1. In `ToyCausalBlock.__init__(self, size)`, build `mask = torch.triu(torch.ones(size, size), diagonal=1).bool()`, then register it as a non-trainable buffer named `"causal_mask"` with `self.register_buffer(...)`.
2. Write `init_weights(module)`: if `isinstance(module, nn.Linear)`, apply `nn.init.xavier_uniform_` to `module.weight` and `nn.init.zeros_` to `module.bias`.

In [ ]:
class ToyCausalBlock(nn.Module):
    def __init__(self, size):
        super().__init__()
        # YOUR CODE HERE
        pass


def init_weights(module):
    # YOUR CODE HERE
    pass

**Verification** — confirm the buffer is in `state_dict()` but not `parameters()`, then re-initialize every `Linear` inside `model_with_shortcut` with a single `.apply()` call.

In [ ]:
block = ToyCausalBlock(4)
print("In parameters():", any(True for _ in block.parameters()))
print("In state_dict():", "causal_mask" in block.state_dict())

first_linear = model_with_shortcut.layers[0][0]
print("\nFirst layer weight (before apply):\n", first_linear.weight)

model_with_shortcut.apply(init_weights)
print("\nFirst layer weight (after apply):\n", first_linear.weight)